# Coding

## Download text file

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !wget https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt -O 10k-most-common.txt

--2022-08-14 15:30:53--  https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 73017 (71K) [text/plain]
Saving to: ‘10k-most-common.txt’

10k-most-common.txt 100%[===================>]  71.31K  --.-KB/s    in 0.01s   

2022-08-14 15:30:53 (5.50 MB/s) - ‘10k-most-common.txt’ saved [73017/73017]



## Example code of 'Hash'

In [3]:
import hashlib

m=hashlib.sha1(b"Chulalongkorn").hexdigest()
m2=hashlib.sha1(b"Chulalongkorn University").hexdigest()
print(m)
print(m2)
m=hashlib.md5(b"Chulalongkorn").hexdigest()
m2=hashlib.md5(b"Chulalongkorn University").hexdigest()
print(m)
print(m2)

ca8a68498ae67cd14c15f5ebf043633224005759
a16c5b03cf3aca5c2f20169b4caa909d5c2f07ad
46fa3b56c660faff420190c18c98a56b
cc3fed293eb73ca7d3597a31259df950


## Implement Word Transformation Function

In [54]:
pair_num_lett = {'o':'0', 'l':'1', 'i': '1', 'O':'0', 'L':'1', 'I': '1'}

def get_all_pos_words(string_word):
  string_word = str(string_word)
  last_uppercase_word = string_word.upper()
  for key, val in pair_num_lett.items():
    last_uppercase_word = last_uppercase_word.replace(key, val)
  all_word_list = [string_word]
  last_word_list = [string_word]
  while last_uppercase_word != last_word_list[-1]:
    new_word_list = set()
    for word in last_word_list:
      for idx in range(len(word)):
        letter = word[idx]
        if letter.islower():
          copy_word = word
          copy_word = copy_word[:idx] + letter.upper() + copy_word[idx+1:]
          new_word_list.add(copy_word)
        if letter in pair_num_lett:
          copy_word = word
          copy_word = copy_word[:idx] + pair_num_lett[letter] + copy_word[idx+1:]
          new_word_list.add(copy_word)
    last_word_list = list(new_word_list)
    all_word_list.extend(last_word_list)
  return all_word_list

## Read '10k-most-common.txt' with Pandas

In [55]:
import time
import pandas as pd

common_words_df = pd.read_csv('/content/10k-most-common.txt', header=None, names=['word'])

## Transform words to all possible cases & Save to txt file

In [56]:
start_time = time.time()

common_words_df['word_list'] = common_words_df["word"].apply(get_all_pos_words)
nest_ls = common_words_df.word_list.values.tolist()
with open('/content/tf_words.txt', "w", encoding="UTF8", newline="") as file:
    for sub_ls in nest_ls:
      for word in sub_ls:
        file.write(word + '\n')

elapsed_time = time.time() - start_time

print(f"elapsed_time of 'Word Transformation' : {elapsed_time}")

elapsed_time of 'Word Transformation' : 33.59225249290466


## Hash passwords & Save to csv file

In [49]:
start_time2 = time.time()

def hash_word(word):
  return hashlib.sha1(str(word).encode()).hexdigest()

password_df = pd.read_csv('/content/tf_words.txt', header=None, names=['password'])
password_df['hashed_password'] = password_df['password'].apply(hash_word)
password_df.to_csv(f'hash_tables.csv', index=False)

elapsed_time2 = time.time() - start_time2

print(f"elapsed_time of 'Hashing passwords' : {elapsed_time2}")

elapsed_time of 'Hashing passwords' : 22.67829704284668


# Q&A

1. Write a simple python program to use the word from the dictionary to find <br> 
the original value of 'd54cc1fe76f5186380a0939d2fc1723c44e8a5f7'. <br>
Note that you might want to include substitution in your code (lowercase, <br>
uppercase, number for letter [‘o’ => 0 , ‘l’ => 1, ‘i’ => 1]). <br>
Hint: Here is a snippet for sha1 and md5 functions.



In [58]:
original_value = 'd54cc1fe76f5186380a0939d2fc1723c44e8a5f7'
original_password = password_df[password_df['hashed_password'] == original_value]['password'].values[0]

print("original_password:", original_password)

original_password is : ThaiLanD


2.  For the given dictionary, create a rainbow table (including the substituted <br>
strings) using the sha1 algorithm. Measure the time for creating such a table. <br>
Measure the size of the table

In [63]:
all_time_spent = elapsed_time + elapsed_time2
print("Time for creating such a table: ", all_time_spent)
print("Size of the table:", len(password_df))

Time for creating such a table:  56.27054953575134
Size of the table: 5004367


3. Based on your code, how long does it take to perform a hash (sha1) on a <br>
password string? Please analyze the performance of your system.


In [73]:
print("A password took about", all_time_spent/len(password_df), "secs.")
print("On the other hand,", int(len(password_df)/all_time_spent),"passwords were successfully hashed per seconds.")

A password took about 1.1244289145011016e-05 secs.
On the other hand, 88934 passwords were successfully hashed per seconds.


4. If you were a hacker obtaining a password file from a system, estimate how <br>
long it takes to break a password with brute force using your computer. <br>
(Please based the answer on your measurement from exercise #3.)


Assuming that
  1. We didn't know the length of the original password (which is 'ThaiLanD').
  2. The characters are 'a-z', 'A-Z', and '0-9'.

In [85]:
print("The number of characters is 26 + 26 + 10 =", 26 + 26 + 10)
original_password_len = len(original_password)
num_possible_passwords = int((62**(original_password_len) - 1)/(original_password_len-1))
print("There are", num_possible_passwords, "possible passwords.")
print()
print("From the performance of this computer,")
time_avg_transforming = elapsed_time/len(password_df)
all_time_transforming = time_avg_transforming*num_possible_passwords
print("it will take", all_time_transforming, "secs to generate all possible passwords,")
time_avg_hashing = elapsed_time2/len(password_df)
all_time_hashing = time_avg_hashing*num_possible_passwords
print("and it will take", all_time_hashing, "secs to hash all possible passwords.")
print()
year_of_bf = (all_time_transforming+all_time_hashing)/60/60/24/365.25
print("Totally, it took around", int(year_of_bf), "years to brute force 'ThaiLanD'.")

The number of characters is 26 + 26 + 10 = 62
There are 31191443654985 possible passwords.

From the performance of this computer,
it will take 209375301.78671253 secs to generate all possible passwords,
and it will take 141350309.52025804 secs to hash all possible passwords.

Totally, it took around 11 years to brute force 'ThaiLanD'.


5. Base on your analysis in exercise #4, what should be the proper length of a <br>
password. (e.g. Take a year or longer to break).


In [94]:
len_selected = 0
for i in range(2,10):
  len_selected = i
  num_pos_pws = int((62**(i) - 1)/(i-1))
  time_tf = time_avg_transforming*num_pos_pws
  time_hash = time_avg_hashing*num_pos_pws
  year_bf = (time_tf+time_hash)/60/60/24/365.25
  if year_bf > 1:
    break

print("The proper length of a password should be at least", len_selected, "characters.")
print("And it took around", int(year_bf), "years to brute force.")

The proper length of a password should be at least 8 characters.
And it took around 11 years to brute force.


6. What is salt? Please explain its role in protecting a password hash.

Salt is a string word which is an addition to an <br> 
original password or a hashed one to extend the time <br> 
of password cracking. As the hacker has to concern <br> 
the index of original password that the salt is added <br> 
and guessing the salt word with more complex <br> 
algorithm, the time of generating possible passwords <br> 
are exponentially increase because of the extended <br> 
length of the password.